In [1]:
import pandas as pd
import json
import os
from collections import Counter

def get_majority_answer(row):
    """
    dev 셋의 5개 정답(answer1~5) 중 가장 많이 나온 최빈값(다수결)을 반환합니다.
    """
    # answer1 부터 answer5까지의 값을 리스트로 추출
    answers = [str(row.get(f'answer{i}', '')).strip() for i in range(1, 6)]
    
    # 비어있는 값이 있다면 제외
    valid_answers = [a for a in answers if a and a != 'nan']
    
    if not valid_answers:
        return ""
        
    # Counter를 이용해 가장 많이 등장한 알파벳 추출
    counter = Counter(valid_answers)
    majority_ans = counter.most_common(1)[0][0]
    
    return majority_ans

def convert_to_vlm_jsonl(input_csv, output_jsonl, is_dev=False):
    """
    CSV 파일을 읽어 Qwen-VL 학습용 JSONL 포맷으로 변환합니다.
    """
    if not os.path.exists(input_csv):
        print(f"⚠️ 파일을 찾을 수 없습니다 패스: {input_csv}")
        return
        
    df = pd.read_csv(input_csv)
    
    # YOLO로 분류되지 않은 모든 데이터(VLM, NaN, 빈칸 등)를 VLM 트랙으로 간주
    vlm_df = df[df['class'].fillna('VLM') != 'YOLO']
    print(f"\n🚀 [{input_csv}] VLM 트랙 데이터 {len(vlm_df)}건 JSONL 변환 시작...")
    
    jsonl_data = []
    
    for idx, row in vlm_df.iterrows():
        img_path = str(row['path'])
        q_text = str(row['question'])
        a_text = str(row.get('a', ''))
        b_text = str(row.get('b', ''))
        c_text = str(row.get('c', ''))
        d_text = str(row.get('d', ''))
        
        # 💡 [핵심] Qwen-VL / LLaMA-Factory 표준 프롬프트 포맷
        # 이미지 경로 태그와 함께 질문과 선택지를 하나의 문자열로 예쁘게 조립합니다.
        user_content = (
            f"<image>{img_path}\n"
            f"질문: {q_text}\n"
            f"a. {a_text}\n"
            f"b. {b_text}\n"
            f"c. {c_text}\n"
            f"d. {d_text}\n"
            f"정답:"
        )
        
        # 정답 추출 (dev 셋은 다수결, train 셋은 단일 정답)
        if is_dev:
            ans = get_majority_answer(row)
        else:
            ans = str(row.get('answer', '')).strip()
            
        # JSON 규격 생성
        record = {
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": ans}
            ]
        }
        jsonl_data.append(record)
        
    # .jsonl 파일로 저장 (한 줄에 JSON 객체 하나씩)
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for item in jsonl_data:
            # ensure_ascii=False: 한글이 유니코드(\u...)로 깨지지 않고 예쁘게 저장되게 함
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print(f"✅ 변환 완료: {output_jsonl} 저장 성공!")

# 실행부: train은 다수결 끔(False), dev는 다수결 켬(True)
convert_to_vlm_jsonl('train_class.csv', 'train_vlm.jsonl', is_dev=False)
convert_to_vlm_jsonl('dev_class.csv', 'dev_vlm.jsonl', is_dev=True)


🚀 [train_class.csv] VLM 트랙 데이터 3326건 JSONL 변환 시작...
✅ 변환 완료: train_vlm.jsonl 저장 성공!

🚀 [dev_class.csv] VLM 트랙 데이터 1269건 JSONL 변환 시작...
✅ 변환 완료: dev_vlm.jsonl 저장 성공!
